In [26]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet("../data/cleaned/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

df.show(5, truncate=False)

+---+-------------------------------+-----------+-------------+------------------------------------------+-----------------------+-------+-------------+--------------------------+-------------------------+---------------------+-----------------+-----------------------------------+
|geo|Geopolitical entity (reporting)|TIME_PERIOD|registrations|manufacturer_name_eu_standard_denomination|commercial_name        |variant|Motor energy |mass_in_running_order (kg)|co2_emissions_WLTP (g/km)|engine_capacity (cm3)|engine_power (KW)|electric_energy_consumption (Wh/km)|
+---+-------------------------------+-----------+-------------+------------------------------------------+-----------------------+-------+-------------+--------------------------+-------------------------+---------------------+-----------------+-----------------------------------+
|DE |Germany                        |2023       |254          |VOLVO                                     |XC40                   |XZBW   |Petrol hybrid|18

Get the number of unique combinations of <'commercial_name', 'engine_capacity (cm3)', and 'engine_power (KW)'> for the consumer for each <'TIME_PERIOD', 'geo', 'Geopolitical entity (reporting)', 'Motor energy'>.

In [27]:
unique_consumer_choices_counts = (
    df
    .groupBy(
        "TIME_PERIOD",
        # "geo",
        # "Geopolitical entity (reporting)",
        "Motor energy"
    )
    .agg(
        F.countDistinct(
            F.struct(
                "commercial_name",
                "engine_capacity (cm3)",
                "engine_power (KW)"
            )
        ).alias("unique_choices")
    )
    .orderBy(
        "TIME_PERIOD",
        # "geo",
        "Motor energy"
    )
)

unique_consumer_choices_counts.show(20, truncate=False)

+-----------+--------------------------+--------------+
|TIME_PERIOD|Motor energy              |unique_choices|
+-----------+--------------------------+--------------+
|2014       |Alternative/Other         |426           |
|2014       |Diesel (excluding hybrids)|7986          |
|2014       |Diesel hybrid             |17            |
|2014       |Electricity               |137           |
|2014       |Petrol (excluding hybrids)|8467          |
|2014       |Petrol hybrid             |122           |
|2015       |Alternative/Other         |360           |
|2015       |Diesel (excluding hybrids)|8223          |
|2015       |Diesel hybrid             |99            |
|2015       |Electricity               |168           |
|2015       |Petrol (excluding hybrids)|8207          |
|2015       |Petrol hybrid             |194           |
|2016       |Alternative/Other         |468           |
|2016       |Diesel (excluding hybrids)|7310          |
|2016       |Diesel hybrid             |27      

In [ ]:
import pyspark.sql.functions as F

by_motor = (
    df
    .groupBy("TIME_PERIOD", "Motor energy")
    .agg(
        F.countDistinct(
            F.struct(
                "commercial_name",
                "engine_capacity (cm3)",
                "engine_power (KW)"
            )
        ).alias("unique_choices"),
        F.sum("registrations").alias("registrations_count")
    )
)

# all_motor = (
#     by_motor
#     .groupBy("TIME_PERIOD")
#     .agg(
#         F.sum("unique_choices").alias("unique_choices"),
#         F.sum("registrations_count").alias("registrations_count")
#     )
#     .withColumn("Motor energy", F.lit("All"))
# )

unique_consumer_choices_counts = (
    by_motor
    # .unionByName(all_motor)
    .withColumn(
        "baseline_normalized_registrations", 
        F.col("unique_choices") / F.col("registrations_count")
    )
    .orderBy("TIME_PERIOD", "Motor energy")
)

unique_consumer_choices_counts.show(20, truncate=False)

+-----------+--------------------------+--------------+-------------------+------------------------+
|TIME_PERIOD|Motor energy              |unique_choices|registrations_count|choices_per_registration|
+-----------+--------------------------+--------------+-------------------+------------------------+
|2014       |Alternative/Other         |426           |236741             |0.0017994348253999096   |
|2014       |Diesel (excluding hybrids)|7986          |5361536            |0.0014894985317640318   |
|2014       |Diesel hybrid             |17            |7217               |0.002355549397256478    |
|2014       |Electricity               |137           |31184              |0.00439327860441252     |
|2014       |Petrol (excluding hybrids)|8467          |4314669            |0.0019623753293705726   |
|2014       |Petrol hybrid             |122           |53408              |0.0022843019772318756   |
|2015       |Alternative/Other         |360           |218485             |0.00164771036913